# Ophthalmic screening prototype - live demo

**Academic screening prototype. Not a medical device and not a diagnosis.** A qualified
clinician reviews every case and makes every decision. Trained on a public research dataset
with 203 cataract and 182 glaucoma patients; the referral documents used in development were
synthetic; there has been no prospective clinical validation.

This runs the **complete system** - the image model, OCR, layout-aware field extraction, the
learned reliability estimator, the concordance check and the deferral rules - and puts it on
a public URL you can share.

Nothing here is reimplemented for the web. It is the same code that produced the reported
results.

**To run it:** `Runtime -> Run all`, then open the URL printed by the last cell. Upload a
fundus photograph, and a referral letter image if you have one.

**The URL lives only while this notebook is running.** Closing the tab or letting Colab time
out takes the demo down with it. That is the trade for it being free and full-speed.


## 1. Unpack the prototype

The runtime bundle is embedded in this notebook, so there is nothing to upload and no repository to clone.

In [ ]:
import base64, io, zipfile, pathlib

BUNDLE = "..."

root = pathlib.Path("/content/screening")
if root.exists():
    import shutil; shutil.rmtree(root)
root.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as archive:
    archive.extractall(root)

print("unpacked to", root)
for path in sorted(root.rglob("*")):
    if path.is_file() and path.suffix in (".pt", ".joblib", ".json", ".py") \
            and "checkpoints" in str(path):
        print(f"  {path.stat().st_size / 1024:8.0f} KB  {path.relative_to(root)}")


## 2. Dependencies

Colab already has torch, torchvision, OpenCV, scikit-learn, pandas and Pillow. Only the OCR stack is missing.

In [ ]:
%pip install -q rapidocr-onnxruntime rapidfuzz

import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)


## 3. Start the app and open a public URL

The server runs in the background. The tunnel is a Cloudflare quick tunnel - no account, no signup.

In [ ]:
import subprocess, time, re, threading, pathlib, urllib.request

ROOT = pathlib.Path("/content/screening")
PORT = 7860

# ODIR images are not redistributed, so there is no pre-screened queue: --cases 0 starts the
# app upload-only, which is the mode a deployed instance runs in anyway.
server = subprocess.Popen(
    ["python", "scripts/serve.py", "--cases", "0", "--no-browser",
     "--port", str(PORT), "--host", "127.0.0.1"],
    cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def echo():
    for line in server.stdout:
        print("[app]", line.rstrip())
threading.Thread(target=echo, daemon=True).start()

for _ in range(120):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3)
        print("\napp is up")
        break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit("the app did not start - see the [app] output above")

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}",
     "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public = None
for line in tunnel.stdout:
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        public = found.group(0)
        break

print("\n" + "=" * 68)
print("  SHARE THIS URL:", public)
print("=" * 68)
print("\n  Upload a fundus photograph, and a referral letter image if you have one.")
print("  The URL stays live only while this notebook is running.")


## 4. While it runs

Leave this notebook open. Colab disconnects idle sessions, which takes the URL down with it.

**What to try.** Upload a fundus photograph on its own and the case is judged on image
confidence alone. Upload one *with* a referral and the document pathway runs too: the fields
are read, their reliability is estimated, and the stated diagnosis is compared against the
image finding. A case is deferred when confidence falls below 60%, when the referral
contradicts the image, or when the fields were read unreliably.

The referral never changes the finding. It only changes whether the system answers.

**To stop:** `Runtime -> Disconnect and delete runtime`, or run the cell below.

In [ ]:
tunnel.terminate(); server.terminate()
print("demo stopped - the public URL is now dead")
